# 4.3 - EWC

Estimate how much each weight mattered for the old tasks (Fisher information), then
penalise moving those weights.

```
loss = CE + lambda * sum_i F_i * (theta_i - theta*_i)^2
```

Memory: Fisher matrix and theta*, no stored data.

Fisher is computed on the current task only, at the end of each task.

Tune `EWC_LAMBDA` from the printed CE and EWC values: EWC should be roughly 5-20% of CE.
Much lower and the penalty does nothing.

## Setup

In [ ]:
import os, sys, subprocess
from getpass import getpass

# Must be set before CUDA initialises, for reproducible cuDNN RNN backward.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(subprocess.run(["git", "-C", ROOT, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USER"
os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_KEY"

from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
%run /content/ulal/src/imports.py

In [ ]:
TASK_CLASSES = [cfg.SELECTED_CLASSES, cfg.TASK_1, cfg.TASK_2, cfg.TASK_4]
TASK_ROOTS   = [cfg.BASE_ROOT, cfg.TASK1_ROOT, cfg.TASK2_ROOT, cfg.TASK4_ROOT]
TASK_NAMES   = ["Base", "Task1", "Task2", "Task4"]
CL_TASKS     = [1, 2]

global_classes, global_c2i, NUM_AFTER = build_label_space(TASK_CLASSES)
describe_label_space(TASK_CLASSES, TASK_NAMES, NUM_AFTER, task_3=cfg.TASK_3)

In [ ]:
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)

In [ ]:
set_seed(cfg.SEED)

# Same for every arm.
CL_EPOCHS, CL_LR, CL_WD, CL_BATCH, CL_LS = 15, 5e-4, 0.03, 32, 0.10
REPLAY_PER_CLASS, REPLAY_STRATEGY        = 15, "hard"

# Reference: best accuracy reachable on these features (Section 2 linear probe).
_d = f"{cfg.RESULTS_DIR}/stage2_embedding_diagnostics.json"
CEILING = json.load(open(_d))["probe"]["JOINT"]["linear"] if os.path.exists(_d) else None
print(f"Ceiling: {CEILING:.2%}" if CEILING else "Ceiling: run Section 2 first")

In [ ]:
# Base head on Task 0. Trained once, then reused by every arm.
base_s, base_y = task_split(CACHE, 0, "train")
val0_s, val0_y = task_split(CACHE, 0, "val")

_ckpt = f"{cfg.CKPT_DIR}/head_base.pt"
base_head = make_temporal_head(NUM_AFTER[0], lstm_state=LSTM_STATE, device=device)

if os.path.exists(_ckpt):
    base_head.load_state_dict(torch.load(_ckpt, map_location=device))
    base_head.eval()
    print("Loaded head_base.pt")
else:
    base_head = finetune_head(
        base_head, base_s, None, base_y, val0_s, val0_y, device,
        epochs=CL_EPOCHS, lr=CL_LR, wd=CL_WD, batch_size=CL_BATCH,
        patience=cfg.FINETUNE_PATIENCE,
        ce_weight=1.0, mse_weight=0.0, label_smoothing=CL_LS,
    )
    os.makedirs(cfg.CKPT_DIR, exist_ok=True)
    torch.save(base_head.state_dict(), _ckpt)
    print("Saved head_base.pt")

print(f"Trainable: {base_head.trainable_params():,}")
print(f"Base test accuracy: {eval_tasks_upto(base_head, CACHE, 0, device)[0]:.2%}")

STREAM = dict(
    base_head=base_head, cache=CACHE, cl_tasks=CL_TASKS, task_classes=TASK_CLASSES,
    task_names=TASK_NAMES, num_after=NUM_AFTER, global_c2i=global_c2i,
    base_classes=cfg.SELECTED_CLASSES, device=device,
    epochs=CL_EPOCHS, lr=CL_LR, wd=CL_WD, batch_size=CL_BATCH, label_smoothing=CL_LS,
)

## Run

In [ ]:
set_seed(cfg.SEED)

EWC_LAMBDA = 1.0

res = run_cl_stream(arm="ewc", use_ewc=True, ewc_lambda=EWC_LAMBDA, **STREAM)

R, m = save_arm_result(
    f"{cfg.RESULTS_DIR}/stage1_arm_ewc.json",
    "EWC", res["rows"], TASK_NAMES, res["bytes_per_class"],
    config={"mechanism": "ewc", "ewc_lambda": EWC_LAMBDA}, ceiling=CEILING,
)
cl_report(R, "EWC", [TASK_NAMES[i] for i in [0] + CL_TASKS])
torch.save(res["head"].state_dict(), f"{cfg.CKPT_DIR}/head_ewc.pt")

## Results so far

In [ ]:
rows = []
for name, fn in [("Naive", "naive"), ("WeightAlign", "weight_align"), ("EWC", "ewc"),
                 ("LwF", "lwf"), ("SmartReplay", "smart_replay"), ("Replay+LwF", "replay_lwf")]:
    fp = f"{cfg.RESULTS_DIR}/stage1_arm_{fn}.json"
    if os.path.exists(fp):
        d = json.load(open(fp))
        rows.append((name, d["metrics"], d["bytes_per_class"]))

print(f"{'Method':<14}{'Base':>9}{'AA':>9}{'BWT':>10}{'Memory/cls':>13}")
print("-" * 55)
for name, m, b in sorted(rows, key=lambda r: r[1]["AA"]):
    mem = "0" if b == 0 else f"{b/1e3:.0f} KB"
    print(f"{name:<14}{m['base_acc_final']:>8.2%}{m['AA']:>9.2%}{m['BWT']:>+10.2%}{mem:>13}")
if CEILING:
    print("-" * 55)
    print(f"{'Ceiling':<14}{'':>9}{CEILING:>9.2%}")